# 2 · HyperSpot preprocessing and hypergraph construction

Creates the HyperSpot project folder (`hyperspot_project_files/Human Kidney Cancer Core/`):
1. **Project metadata**: cell-type key and list, used by every later step.
2. **Preprocessed data**: the Visium data (`ST_orig`) and the decomposed cells (`ST_decomposition`).
3. **Per-cell-type data**: one AnnData per cell type with its own top-3000 feature genes.
4. **Hypergraph**: inferred cells are nodes; each spot plus its neighbouring spots forms a hyperedge.

Kernel: `HyperSpot`. Next: `03 HyperSpot Pipeline`.

In [ ]:
import os
import scanpy as sc

from hyperspot.metadata import ProjectMetadata
from hyperspot.preprocessing import common, singlecell, visium
from hyperspot.utils import save_orig_st_data, save_decomposed_st_data

from pathlib import Path

# All tutorial outputs live next to these notebooks; run the notebooks from tutorial/.
DATA_DIR      = Path("../demo/data_repo/Human Kidney Cancer")     # raw inputs (see README)
PATIENT_ID    = "PD47171"
PROJECT       = "Human Kidney Cancer Core"
SPOTIPHY_DIR  = Path("spotiphy_project_files") / PROJECT          # Spotiphy inputs/outputs
HYPERSPOT_DIR = Path("hyperspot_project_files") / PROJECT         # HyperSpot project (the CLI resolves
                                                                  # --project relative to ./hyperspot_project_files)
HYPERSPOT_DIR.mkdir(parents=True, exist_ok=True)

## Load the Spotiphy outputs

In [ ]:
adata_decomp = sc.read_h5ad(SPOTIPHY_DIR / "ST_decomposition.h5ad")
adata_st     = sc.read_h5ad(SPOTIPHY_DIR / f"{PROJECT}.h5ad")

# attach Spotiphy's generated per-cell coordinates (from segmentation.pkl)
adata_decomp = common.add_generated_cell_locations(SPOTIPHY_DIR, adata_decomp, adata_st)
adata_decomp

## Project metadata

`ProjectMetadata.create` records the project folder, the cell-type key (`cell_type`), the sorted
cell-type list and the folder layout. Later steps and the CLI read it from the project folder.

In [ ]:
project_metadata = ProjectMetadata.create(HYPERSPOT_DIR, adata_decomp, "cell_type")
project_metadata.ct_list

## Filter and normalise

Mitochondrial genes are removed before embedding. The Visium data is log-CPM normalised; the raw
counts stay in `.raw` for plotting.

In [ ]:
MT_IDS = ["ENSG00000198886", "ENSG00000198888", "ENSG00000228253", "ENSG00000198899",
          "ENSG00000198695", "ENSG00000198840", "ENSG00000198938", "ENSG00000198804",
          "ENSG00000198712", "ENSG00000212907", "ENSG00000198763", "ENSG00000198727",
          "ENSG00000198786"]
is_mt_gene = lambda a: a.var_names.isin(MT_IDS)

adata_decomp = common.filter(adata_decomp, min_genes=20, min_locs=20, is_mt_gene=is_mt_gene)
adata_st     = common.filter(adata_st,     min_genes=20, min_locs=20, is_mt_gene=is_mt_gene)

N_FEATURES = 3000
adata_st.raw = adata_st.copy()
sc.pp.highly_variable_genes(adata_st, flavor="seurat_v3", n_top_genes=N_FEATURES)
sc.pp.normalize_total(adata_st, target_sum=1e6)
sc.pp.log1p(adata_st)

save_orig_st_data(adata_st)
save_decomposed_st_data(adata_decomp)

## Per-cell-type data and feature genes

Splits the decomposed cells by type, normalises each type separately, and picks each type's top
`N_FEATURES` genes. These are the model's per-type input features.

In [ ]:
singlecell.make_ct_spatial_adata(adata_decomp)
singlecell.perform_ct_feature_selection(n_features=N_FEATURES)

In [ ]:
from hyperspot.visualization import plot_all_ct_decomposition
plot_all_ct_decomposition(n_cols=6)

## Build the hypergraph

- **Nodes**: inferred cells, with their cell type's feature genes.
- **Hyperedges**: one per spot, joining the cells of that spot and of its `num_neighbor_spots`
  nearest spots (7 = the spot plus its 6 hexagonal neighbours).
- **Hyperedge features**: the spot's pseudobulk over `n_hyperedge_feats` genes.

In [ ]:
hgdata = visium.create_hypergraph_data(adata_st, n_features=N_FEATURES,
                                       num_neighbor_spots=7, n_hyperedge_feats=N_FEATURES)
hgdata.save()
sorted(p.name for p in HYPERSPOT_DIR.iterdir())